In [19]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [24]:
import os
os.listdir('/content/drive/MyDrive/Colab')

['placement_dataset.csv', 'placement_predict_50k.csv']

In [25]:
import pandas as pd
df = pd.read_csv('/content/drive/MyDrive/Colab/placement_predict_50k.csv')

In [26]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score

# ---------------------------------------------------------
# 1. Load data
# ---------------------------------------------------------
df = pd.read_csv('/content/drive/MyDrive/Colab/placement_predict_50k.csv')

target_col = "CGPA"

# Drop the ID column and CGPA_Tier (a label derived directly from CGPA,
# so keeping it would leak the target) before selecting numeric features
drop_cols = [target_col, "StudentID", "CGPA_Tier"]
feature_df = df.drop(columns=drop_cols).select_dtypes(include=[np.number])

# Some columns (Workshops, AptitudeTestScore, SoftSkillsRating,
# CodingTestScore, MockInterviewScore) have missing values —
# impute with the column median so gradient descent doesn't blow up on NaNs
feature_df = feature_df.fillna(feature_df.median())

X = feature_df.values
y = df[target_col].values.reshape(-1, 1)

# ---------------------------------------------------------
# 2. Train/test split
# ---------------------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# ---------------------------------------------------------
# 3. Feature scaling (helps gradient descent converge)
# ---------------------------------------------------------
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Add bias column (intercept term)
X_train_b = np.c_[np.ones((X_train.shape[0], 1)), X_train]
X_test_b = np.c_[np.ones((X_test.shape[0], 1)), X_test]

# ---------------------------------------------------------
# 4. Gradient Descent implementation
# ---------------------------------------------------------
def gradient_descent(X, y, lr=0.01, n_iters=1000):
    m, n = X.shape
    theta = np.zeros((n, 1))
    losses = []

    for i in range(n_iters):
        y_pred = X @ theta
        error = y_pred - y

        # Mean Squared Error loss
        loss = (1 / (2 * m)) * np.sum(error ** 2)
        losses.append(loss)

        # Gradient of MSE w.r.t. theta
        gradient = (1 / m) * (X.T @ error)

        # Update rule
        theta -= lr * gradient

        if i % 100 == 0:
            print(f"Iteration {i:4d} | Loss: {loss:.4f}")

    return theta, losses

theta, losses = gradient_descent(X_train_b, y_train, lr=0.01, n_iters=1000)

# ---------------------------------------------------------
# 5. Evaluate
# ---------------------------------------------------------
y_pred = X_test_b @ theta

mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("\nFinal weights (theta):")
print(theta.ravel())
print(f"\nTest MSE : {mse:.4f}")
print(f"Test R^2 : {r2:.4f}")

# ---------------------------------------------------------
# 6. (Optional) Compare with sklearn's LinearRegression
# ---------------------------------------------------------
from sklearn.linear_model import LinearRegression

sk_model = LinearRegression()
sk_model.fit(X_train, y_train)
sk_pred = sk_model.predict(X_test)

print(f"\nsklearn LinearRegression R^2: {r2_score(y_test, sk_pred):.4f}")

Iteration    0 | Loss: 27.5449
Iteration  100 | Loss: 3.6438
Iteration  200 | Loss: 0.5800
Iteration  300 | Loss: 0.1654
Iteration  400 | Loss: 0.1082
Iteration  500 | Loss: 0.0998
Iteration  600 | Loss: 0.0983
Iteration  700 | Loss: 0.0979
Iteration  800 | Loss: 0.0977
Iteration  900 | Loss: 0.0976

Final weights (theta):
[ 7.24382401  0.22136797  0.20714545  0.19391788  0.18710597  0.1703889
  0.15718881  0.15590153  0.14210222  0.04466781  0.07018233  0.09508345
 -0.09890673 -0.13948908 -0.06433497  0.21609615 -0.09399594  0.37238503
 -0.17019913 -0.02192028 -0.10237805 -0.03382736  0.06590428]

Test MSE : 0.2051
Test R^2 : 0.9212

sklearn LinearRegression R^2: 0.9216
